# A3 — Saneamento Inteligente: inspeção da base integrada

**Base integrada, referência 2024.** Este notebook reúne a inspeção dos dados, validações e descritivas iniciais. O diagnóstico, as hipóteses e as prescrições estão consolidados no relatório e nas análises do projeto.

O tema foi informado como aprovado pelo professor. Os requisitos conferidos estão em `docs/REQUISITOS_VERIFICADOS.md`; os anexos originais foram preservados no pacote local.

Para reproduzir da raiz do projeto: `python scripts/preparar_dados.py --download` e `python scripts/eda_inicial.py`. A opção `--download` busca os arquivos oficiais SINISA, IBGE e as tabelas RAIS. O agregado setorial RAIS validado acompanha o projeto; para reler seus microdados, use também `--reprocessar-setorial`, conforme `docs/RAIS_SETORIAL.md`.

Não é necessário abrir as planilhas manualmente nem alterar seus cabeçalhos.


In [1]:
from pathlib import Path
import json
import pandas as pd

# Funciona quando o notebook é aberto na raiz do projeto ou na pasta notebooks.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'outputs/saneamento-inteligente-a3']
ROOT = next(p for p in candidates if (p / 'data/processed/saneamento_municipios_2024_parcial.csv').exists())
df = pd.read_csv(ROOT / 'data/processed/saneamento_municipios_2024_parcial.csv', dtype={'cod_ibge': str})
validation = json.loads((ROOT / 'evidencias/validacoes_etl.json').read_text(encoding='utf-8'))
print('Linhas e colunas:', df.shape)
print('Fontes efetivamente integradas:', validation['fontes_realmente_integradas'])
print('Uma linha por município e ano:', not df[['cod_ibge', 'ano']].duplicated().any())


Linhas e colunas: (5570, 56)
Fontes efetivamente integradas: ['SINISA', 'IBGE', 'RAIS']
Uma linha por município e ano: True


## 1. O que significa integrar as fontes

O código IBGE de sete dígitos identifica o município. O ETL verifica correspondência um a um e UF. Uma terceira fonte apenas localizada na internet não conta como integrada. O campo `fontes_integradas` informa disponibilidade por linha; ausência de resposta no SINISA continua indicada em coluna própria.

SINISA 2025 descreve o ano 2024. Para IBGE, usamos a publicação DOU de 29/08/2024: suas 5.570 populações coincidem com a base demográfica do SINISA. A revisão posterior está preservada, mas altera o recorte territorial.


In [2]:
columns = ['cod_ibge', 'municipio', 'uf', 'ano', 'populacao_ibge_2024',
           'respondeu_sinisa', 'despesa_exploracao_manejo_rs_ano',
           'massa_rsu_coletada_t_ano', 'massa_recuperada_secos_organicos_t_ano',
           'status_rais']
print(df[columns].head(8).to_string(index=False))


cod_ibge             municipio uf  ano  populacao_ibge_2024 respondeu_sinisa  despesa_exploracao_manejo_rs_ano  massa_rsu_coletada_t_ano  massa_recuperada_secos_organicos_t_ano                                         status_rais
 1100015 Alta Floresta D'Oeste RO 2024                22853              Sim                         530000.00                    4148.0                                     0.0 CNAE 38 disponível; aproximação setorial de emprego
 1100023             Ariquemes RO 2024               108573              Sim                        6459926.05                   75812.5                                   961.6 CNAE 38 disponível; aproximação setorial de emprego
 1100031                Cabixi RO 2024                 5690              Sim                         762769.86                     656.7                                     0.0 CNAE 38 disponível; aproximação setorial de emprego
 1100049                Cacoal RO 2024                97637              Sim        

## 2. Ausente não é zero

Os não respondentes não foram eliminados do universo nem receberam valores zero. Para analisar despesas e recuperação, usamos inicialmente respondentes com denominadores positivos e sem os alertas já detectados. Outras verificações contextuais ainda são necessárias.


In [3]:
print('Respostas ao SINISA:')
print(df['respondeu_sinisa'].value_counts(dropna=False).to_string())
print('\nAusências nas variáveis selecionadas:')
print(df[columns].isna().sum().to_string())
print('\nNúmero de diferenças de população SINISA x IBGE:', validation['diferencas_populacao_sinisa_ibge'])


Respostas ao SINISA:
respondeu_sinisa
Sim    4854
Não     716

Ausências nas variáveis selecionadas:
cod_ibge                                    0
municipio                                   0
uf                                          0
ano                                         0
populacao_ibge_2024                         0
respondeu_sinisa                            0
despesa_exploracao_manejo_rs_ano          716
massa_rsu_coletada_t_ano                  716
massa_recuperada_secos_organicos_t_ano    716
status_rais                                 0

Número de diferenças de população SINISA x IBGE: 0


## 3. Verificações e decisões

Preservamos os registros com problemas e os marcamos. Não corrigimos automaticamente valores oficiais sem evidência. Compare GTR1028 com GTR1025 + GTR1027, percentuais, denominadores e o indicador oficial IRS3010.


In [4]:
print('Alertas por regra:')
print(pd.Series(validation['alertas_por_regra']).to_string())
print('\nMunicípios sinalizados:')
show = ['cod_ibge', 'municipio', 'uf', 'massa_rsu_coletada_t_ano',
        'massa_domiciliar_coletada_t_ano', 'massa_limpeza_urbana_t_ano']
print(df.loc[df.flag_qualidade, show].to_string(index=False))


Alertas por regra:
cobertura_coleta_sinisa_oficial_pct_fora_0_100          0
cobertura_seletiva_sinisa_oficial_pct_fora_0_100        0
recuperacao_total_sinisa_oficial_pct_fora_0_100         0
recuperacao_secos_sinisa_oficial_pct_fora_0_100         0
recuperacao_organicos_sinisa_oficial_pct_fora_0_100     0
disposicao_inadequada_sinisa_oficial_pct_fora_0_100     0
recuperacao_total_calculada_pct_fora_0_100              0
cobertura_coleta_ibge2024_pct_fora_0_100                0
cobertura_seletiva_ibge2024_pct_fora_0_100              0
participacao_cnae38_no_emprego_formal_pct_fora_0_100    0
despesa_manejo_nao_positiva                             0
massa_rsu_nao_positiva                                  0
massa_rsu_difere_domiciliar_mais_limpeza                2
recuperacao_diverge_oficial_mais_0_02_pp                0

Municípios sinalizados:
cod_ibge       municipio uf  massa_rsu_coletada_t_ano  massa_domiciliar_coletada_t_ano  massa_limpeza_urbana_t_ano
 3305133 São José de Ubá RJ  

## 4. Primeiras estatísticas — leitura cautelosa

A despesa abaixo é GFI2221 por população **residente**, enquanto IFR2002 oficial usa população **coberta pela coleta**. Recuperação inclui secos e orgânicos, segundo a definição SINISA. Medianas municipais não são taxas nacionais ponderadas.

Despesa por tonelada abrange o manejo e não representa o preço exclusivo da coleta. Não chamamos menor despesa automaticamente de maior eficiência.


In [5]:
sample = df.loc[df.apta_analise_custo_recuperacao].copy()
metrics = ['despesa_manejo_por_hab_ibge2024', 'despesa_manejo_por_t_rsu',
           'recuperacao_total_calculada_pct', 'cobertura_seletiva_sinisa_oficial_pct']
if 'vinculos_formais_totais_por_1000_hab' in sample:
    metrics.append('vinculos_formais_totais_por_1000_hab')
print('Amostra municipal preliminar:', len(sample))
print(sample[metrics].describe().T.to_string())
print('\nPor região — cobertura de resposta:')
coverage = df.groupby('regiao').respondeu_sinisa.agg(municipios='size', respondentes=lambda x: x.eq('Sim').sum())
print(coverage.to_string())


Amostra municipal preliminar: 4852
                                        count        mean           std       min         25%         50%         75%            max
despesa_manejo_por_hab_ibge2024        4852.0  232.347412   3230.527553  0.080553   31.346486   69.166143  121.581092  121860.465116
despesa_manejo_por_t_rsu               4852.0  915.413993  12459.914591  0.151429  109.232310  262.045613  512.032860  554078.549849
recuperacao_total_calculada_pct        4852.0    2.808174      7.414021  0.000000    0.000000    0.000000    1.390021      99.734043
cobertura_seletiva_sinisa_oficial_pct  4852.0   20.802566     35.930441  0.000000    0.000000    0.000000   33.857500     100.000000
vinculos_formais_totais_por_1000_hab   4848.0  178.745742    111.524586  3.365689   96.617105  156.814910  232.830235    1266.163793

Por região — cobertura de resposta:
              municipios  respondentes
regiao                                
Centro-Oeste         467           424
Nordeste     

![Gráficos preliminares](../analises/eda_preliminar.png)

O histograma e a dispersão usam escala log para tornar a distribuição legível. O box plot oculta somente a exibição de pontos extremos; eles permanecem no dataset. Os gráficos e as tabelas são gerados por `scripts/eda_inicial.py`.

## 5. Questões examinadas e continuidade

- A diferença de despesas permanece quando comparamos portes e regiões semelhantes?
- Os maiores valores são explicados por escopo dos serviços, distâncias, erros ou condições locais?
- A cobertura seletiva está associada à recuperação? A conclusão muda ao separar secos e orgânicos?
- Com a RAIS setorial validada, há associação com vínculos formais? Catadores informais não são medidos por essa fonte.

Correlação não demonstra causalidade. Um único ano não permite afirmar tendência ou criação de empregos.

## Fontes oficiais e rastreabilidade

- [SINISA 2025, referência 2024](https://www.gov.br/cidades/pt-br/acesso-a-informacao/acoes-e-programas/saneamento/sinisa/resultados-sinisa/resultados-sinisa-2025)
- [IBGE — arquivos 2024](https://ftp.ibge.gov.br/Estimativas_de_Populacao/Estimativas_2024/)
- [MTE — RAIS 2024](https://www.gov.br/trabalho-e-emprego/pt-br/assuntos/estatisticas-trabalho/rais/rais-2024)

Consulte o inventário de fontes, os dicionários e o registro de metodologia em `docs/`. Consulte `evidencias/validacoes_etl.json` para a execução efetiva. O diagnóstico, as prescrições e os materiais de apresentação estão nas pastas próprias do projeto. Os resultados das hipóteses estão em `docs/HIPOTESES_E_QUESTOES.md`. O [painel online](https://a3-saneamento-inteligente-brasil.streamlit.app/) e o [repositório](https://github.com/herbarth/a3-saneamento-inteligente) foram publicados e validados. Identificação dos integrantes e vídeo final permanecem pendentes no acompanhamento da A3.
